In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# This tells the website we're a regular browser
HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'
}

# The URL you gave me
url = "https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD"

# Try to fetch the page
print("Fetching the page...")
response = requests.get(url, headers=HEADERS)
print(f"Status code: {response.status_code}")  # 200 means success!

# Look at the first 500 characters of HTML
print("\nFirst 500 characters of HTML:")
print(response.text[:500])

Fetching the page...
Status code: 200

First 500 characters of HTML:
<!doctype html>
<html lang="en">


<head>

    
    <title>Search | Legislative Observatory | European Parliament</title>

    <!-- Required meta tags -->
    <meta charset="utf-8">
    <meta content="@project.version@" name="project_version"/>
    <meta content="@build.number@" name="build_number"/>
    <meta content="@build.time@" name="build_time"/>

    <!-- Commons meta tags -->
    <meta content="https://www.europarl.europa.eu">
    <meta name="viewport" content="width=device-width, initia


In [2]:
# Parse the HTML
soup = BeautifulSoup(response.content, 'html.parser')

# Find all links that go to procedure pages
procedure_links = []
for link in soup.find_all('a', href=True):
    href = link['href']
    if '/procedure-file?reference=' in href:
        full_url = "https://oeil.europarl.europa.eu" + href
        procedure_links.append(full_url)

# Remove duplicates (just in case)
procedure_links = list(set(procedure_links))

print(f"Found {len(procedure_links)} procedure links on the first page!")
print("\nFirst 5 links:")
for link in procedure_links[:5]:
    print(link)

Found 30 procedure links on the first page!

First 5 links:
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0400(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0417(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2026/0011(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2026/0013(COD)
https://oeil.europarl.europa.eu/oeil/en/procedure-file?reference=2025/0409(COD)


In [ ]:
def parse_search_results(html):
    """Extract all available information from search page HTML"""
    soup = BeautifulSoup(html, 'html.parser')
    procedures = []
    
    for doc in soup.find_all('div', class_='es_document-wrapper'):
        proc = {}
        
        # Find the main title/header area
        title_tag = doc.find('h3', class_='es_document-title')
        if title_tag:
            link_tag = title_tag.find('a')
            if link_tag:
                proc['url'] = "https://oeil.europarl.europa.eu" + link_tag['href']
                # Extract reference from URL
                import re
                match = re.search(r'reference=([^&]+)', link_tag['href'])
                if match:
                    proc['reference'] = match.group(1)
            
            # Extract committees from badges inside the title
            committees = []
            badge_spans = title_tag.find_all('span', class_='es_badge-committee')
            for badge in badge_spans:
                committees.append(badge.get_text(strip=True))
            if committees:
                proc['committees'] = ', '.join(committees)
        
        # Get the title text
        body = doc.find('div', class_='es_document-body')
        if body:
            title_p = body.find('p')
            if title_p:
                proc['title'] = title_p.get_text(strip=True)
        
        # Extract rapporteur(s) - improved
        rapporteur_div = doc.find('div', class_='es_document-subtitle', 
                                string=lambda text: text and 'Rapporteur' in text)
        if rapporteur_div:
            # Find all author spans
            author_spans = rapporteur_div.find_all('span', class_='es_document-subtitle-author')
            rapporteurs = [span.get_text(strip=True) for span in author_spans]
            if rapporteurs:
                proc['rapporteur'] = '; '.join(rapporteurs)
        
        # Only add if we found something
        if proc:
            procedures.append(proc)
    
    return procedures

In [ ]:
def get_total_results_from_page(html):
    soup = BeautifulSoup(html, 'html.parser')
    total_span = soup.find('span', class_='document-search-total-results-count')
    if total_span:
        # Remove commas and convert to int
        total_text = total_span.get_text(strip=True).replace(',', '')
        return int(total_text)
        print(int(total_text))
    return None

In [ ]:
def fetch_all_pages():
    all_procedures = []
    page = 1
    results_per_page = 30
    
    # First, fetch page 1 to get total
    first_url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex=1"
    print("Fetching first page to get total...")
    response = requests.get(first_url, headers=HEADERS, timeout=10)
    response.raise_for_status()
    
    total_results = get_total_results_from_page(response.text)
    if total_results is None:
        print("Could not find total results count. Using hardcoded 22062.")
        total_results = 22062
    
    total_pages = (total_results + results_per_page - 1) // results_per_page
    print(f"Total results: {total_results}, total pages: {total_pages}")
    
    # Parse first page
    first_page_results = parse_search_results(response.text)
    all_procedures.extend(first_page_results)
    print(f"  Page 1: found {len(first_page_results)} procedures")
    
    # Loop for remaining pages
    page = 2
    while page <= total_pages:
        print(f"Fetching page {page}/{total_pages}...")
        url = f"https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex={page}"
        
        try:
            response = requests.get(url, headers=HEADERS, timeout=10)
            response.raise_for_status()
            
            page_results = parse_search_results(response.text)
            all_procedures.extend(page_results)
            
            print(f"  Found {len(page_results)} procedures on this page")
            
            page += 1
            time.sleep(1)
            
        except Exception as e:
            print(f"Error on page {page}: {e}")
            break
    
    return all_procedures

In [ ]:
# Test on first page
url = "https://oeil.europarl.europa.eu/oeil/en/search?fullText.mode=EXACT_WORD&pageIndex=1"
response = requests.get(url, headers=HEADERS)
first_page_results = parse_search_results(response.text)

# Check how many have rapporteur
with_rapporteur = [p for p in first_page_results if 'rapporteur' in p]
print(f"Total on first page: {len(first_page_results)}")
print(f"With rapporteur: {len(with_rapporteur)}")
print("Sample with rapporteur:")
for p in with_rapporteur[:3]:
    print(p.get('reference'), p.get('rapporteur'))

In [ ]:
print("Starting to fetch all procedures...")
all_data = fetch_all_pages()
print(f"\nTotal procedures collected: {len(all_data)}")

# Create DataFrame
df_all = pd.DataFrame(all_data)
print(f"DataFrame shape: {df_all.shape}")
print("\nFirst 5 rows:")
print(df_all.head())

# # Save to CSV
# df_all.to_csv('europarl_all_procedures.csv', index=False)
# print("\nSaved all data to 'europarl_all_procedures.csv'")

In [ ]:
df_all

In [ ]:
df_all.to_csv('oeil_all_procedures_landing_page.csv', index=False)

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
from typing import List, Optional, Union, Tuple

# ---------- existing functions (unchanged, kept for reference) ----------
def fetch_html(url: str) -> str:
    response = requests.get(url)
    response.raise_for_status()
    return response.text

def find_sections_by_header(soup: BeautifulSoup, header_text: str) -> List[BeautifulSoup]:
    sections = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        next_div = parent.find_next('div', class_='table-responsive')
        if next_div and next_div not in sections:
            sections.append(next_div)
    return sections

def extract_committee_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> Optional[str]:
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return None
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            p = cells[1].find('p')
            if p:
                return p.get_text(strip=True)
            return cells[1].get_text(strip=True)
    return None

def count_rollcall_sections(soup: BeautifulSoup, header_text: str) -> int:
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    return len(list(soup.find_all(string=pattern)))

# ---------- new helper functions ----------
def find_all_divs_after_header(soup: BeautifulSoup, header_text: str, class_name: str = 'table-responsive') -> List[BeautifulSoup]:
    """
    Find all divs with class `class_name` that appear after an element containing header_text.
    Returns a list of those divs.
    """
    divs = []
    pattern = re.compile(re.escape(header_text), re.IGNORECASE)
    for elem in soup.find_all(string=pattern):
        parent = elem.parent
        while parent and parent.name not in ['h1','h2','h3','h4','h5','h6','p','div','section']:
            parent = parent.parent
        if not parent:
            parent = elem.parent
        # find all following divs with the given class
        for sibling in parent.find_all_next('div', class_=class_name):
            # stop if we hit another header? For simplicity we collect all until end of section.
            # But the page structure is consistent: all such divs under the same heading belong to it.
            if sibling not in divs:
                divs.append(sibling)
    return divs

def extract_committees_from_table(procedure_div: BeautifulSoup, label_texts: Union[str, List[str]]) -> List[str]:
    """
    Extract all committee codes from a procedure table.
    Finds the row whose first cell contains any of the label_texts (case‑insensitive),
    then collects the text from all subsequent cells (or their first paragraphs).
    Returns a list of committee codes.
    """
    if isinstance(label_texts, str):
        label_texts = [label_texts]
    table = procedure_div.find('table')
    if not table:
        return []
    rows = table.find_all('tr')
    for row in rows:
        cells = row.find_all('td')
        if len(cells) < 2:
            continue
        label = cells[0].get_text(strip=True)
        if any(lt.lower() in label.lower() for lt in label_texts):
            committees = []
            # start from cell 1 (index 1) because cell 0 is the label
            for cell in cells[1:]:
                # committee code is usually in the first paragraph, otherwise whole cell text
                p = cell.find('p')
                text = p.get_text(strip=True) if p else cell.get_text(strip=True)
                if text:  # ignore empty cells
                    committees.append(text)
            return committees
    return []

# ---------- new main function ----------
def get_committee_votes_with_html(url: str, procedure_number: str) -> pd.DataFrame:
    """
    Fetch the HTML from the given Europarl document URL and extract
    information about responsible and opinion committees, including the
    full HTML of the procedure and roll‑call tables.

    Parameters
    ----------
    url : str
        The URL of the document (e.g. https://www.europarl.europa.eu/doceo/document/...)
    procedure_number : str
        The procedure identifier, e.g. '2021/0366(COD)'

    Returns
    -------
    pd.DataFrame
        Columns: procedure, committee, designation, procedure_available,
                 rollcall_available, procedure_string, rollcall_string
    """
    html = fetch_html(url)
    soup = BeautifulSoup(html, 'html.parser')

    # ----- Check for "document not in HTML" placeholder -----
    placeholder = soup.select_one('div.card.mb-3 div.card-body p.bg-white.alert.alert-info.mb-3')
    if placeholder and "This document does not exist in HTML but is available in the following formats:" in placeholder.get_text():
        return pd.DataFrame([{
            'procedure': procedure_number,
            'committee': 'other format',
            'designation': 'other format',
            'procedure_available': False,
            'rollcall_available': False,
            'procedure_string': '',
            'rollcall_string': ''
        }])

    # ---- Collect responsible committee and its procedure table ----
    resp_procedure_div = None
    resp_committee = None

    # Try specific header first
    resp_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE RESPONSIBLE")
    for div in resp_sections:
        code = extract_committee_from_table(div, "Committee responsible")
        if code:
            resp_committee = code
            resp_procedure_div = div
            break

    # If not found, try generic "PROCEDURE"
    if not resp_committee:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        for div in generic_sections:
            code = extract_committee_from_table(div, "Committee responsible")
            if code:
                resp_committee = code
                resp_procedure_div = div
                break

    # ---- Collect responsible roll‑call tables ----
    resp_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE RESPONSIBLE")
    resp_rollcall_exists = len(resp_rollcall_divs) > 0

    # ---- Collect opinion committees with their source tables ----
    # We'll store tuples (committee_code, procedure_div)
    opinion_entries = []

    # 1. Sections with header "PROCEDURE – COMMITTEE ASKED FOR OPINION"
    opinion_sections = find_sections_by_header(soup, "PROCEDURE – COMMITTEE ASKED FOR OPINION")
    for div in opinion_sections:
        codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
        for code in codes:
            opinion_entries.append((code, div))

    # 2. Sections with header "OPINION"
    if not opinion_entries:
        opinion_headers_sections = find_sections_by_header(soup, "OPINION")
        for div in opinion_headers_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                opinion_entries.append((code, div))

    # 3. Generic "PROCEDURE" sections (only if we haven't found any yet, and avoid duplicates)
    if not opinion_entries:
        generic_sections = find_sections_by_header(soup, "PROCEDURE")
        existing = set()
        for div in generic_sections:
            codes = extract_committees_from_table(div, ["Opinion by", "Committee(s) asked for opinion(s)"])
            for code in codes:
                if code.upper() not in existing:
                    opinion_entries.append((code, div))
                    existing.add(code.upper())

    # ---- Collect opinion roll‑call tables ----
    opinion_rollcall_divs = find_all_divs_after_header(soup, "FINAL VOTE BY ROLL CALL IN COMMITTEE ASKED FOR OPINION")
    opinion_rollcall_count = len(opinion_rollcall_divs)

    # ---- Build rows ----
    rows = []

    # Responsible committee row
    if resp_committee:
        proc_html = str(resp_procedure_div) if resp_procedure_div else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': resp_committee,
            'designation': 'lead',
            'procedure_available': resp_procedure_div is not None,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })
    elif resp_sections:  # section exists but no committee code found
        proc_html = str(resp_sections[0]) if resp_sections else ''
        roll_html = ''.join(str(div) for div in resp_rollcall_divs) if resp_rollcall_exists else ''
        rows.append({
            'procedure': procedure_number,
            'committee': 'UNKNOWN',
            'designation': 'lead',
            'procedure_available': True,
            'rollcall_available': resp_rollcall_exists,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    # Opinion committee rows
    for i, (comm, proc_div) in enumerate(opinion_entries):
        proc_html = str(proc_div) if proc_div else ''
        # rollcall available only for the first `opinion_rollcall_count` committees
        roll_avail = i < opinion_rollcall_count
        roll_html = ''.join(str(div) for div in opinion_rollcall_divs) if roll_avail else ''
        rows.append({
            'procedure': procedure_number,
            'committee': comm,
            'designation': 'opinion',
            'procedure_available': True,  # we have a procedure div
            'rollcall_available': roll_avail,
            'procedure_string': proc_html,
            'rollcall_string': roll_html
        })

    return pd.DataFrame(rows)

In [8]:
df = get_committee_votes_with_html(
    url='https://www.europarl.europa.eu/doceo/document/A-4-1995-0221_EN.html',
    procedure_number='2021/0366(COD)'
)

df

,procedure,committee,designation,procedure_available,rollcall_available,procedure_string,rollcall_string
0,2021/0366(COD),other format,other format,False,False,,


In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_procedure_table(html_str: str, committee_type: str = 'lead') -> list:
    """
    Parse the procedure table HTML and extract relevant rows.

    Parameters
    ----------
    html_str : str
        The HTML content of the div.table-responsive containing the procedure table.
    committee_type : str
        Either 'lead' or 'opinion'. Determines which header texts to look for.

    Returns
    -------
    list of dict
        Each dict has keys 'head' and 'value'. The head is the cleaned left cell text.
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return []

    rows = table.find_all('tr')
    extracted = []

    # Mapping of expected head texts for lead vs opinion
    # We'll use these to decide how to parse each row
    for tr in rows:
        cells = tr.find_all(['td', 'th'])
        if not cells:
            continue

        # First cell is the head / label
        head_cell = cells[0]
        head = head_cell.get_text(strip=True)
        if not head:
            continue

        # Normalise head (strip, lower for matching, but keep original for output)
        head_lower = head.lower()

        # Skip rows that are not of interest (like empty or just formatting)
        # But we'll keep all and filter later; better to only process known heads.
        # However, the user may want all rows; we'll process all and let the caller decide.
        # We'll handle special cases based on head text.

        # Determine number of value cells
        value_cells = cells[1:]

        # Special handling based on head
        if 'result of final vote' in head_lower:
            # Expect pattern: +: number, -: number, 0: number interleaved
            # Iterate through value cells, treat pairs (label, number)
            i = 0
            while i < len(value_cells):
                cell = value_cells[i]
                text = cell.get_text(strip=True)
                # Check if this cell contains a vote symbol
                if text in ['+:', '–:', '-:', '0:']:
                    # Next cell should contain the number
                    if i+1 < len(value_cells):
                        num_cell = value_cells[i+1]
                        num_text = num_cell.get_text(strip=True)
                        # Determine head suffix
                        if text in ['+:', '+']:
                            suffix = 'pos'
                        elif text in ['–:', '-:']:
                            suffix = 'neg'
                        elif text == '0:':
                            suffix = 'abs'
                        else:
                            suffix = text.strip(':')
                        extracted.append({
                            'head': f'Result {suffix}',
                            'value': num_text
                        })
                        i += 2
                    else:
                        i += 1
                else:
                    # No symbol, just treat as normal? Should not happen.
                    # But if the numbers are all in one cell? In example they are separate.
                    # We'll skip or handle as fallback.
                    i += 1
            continue  # already added rows

        elif 'rapporteur' in head_lower and 'opinion' not in head_lower:
            # For lead: "Rapporteurs" (maybe multiple rapporteurs? But usually one)
            # Expect first value cell = name, second = date
            # If there are multiple rapporteurs? Unlikely, but could be.
            # We'll extract all non-empty cells, assume alternating name/date? Not reliable.
            # Simpler: treat first value cell as name, second as date (if exists)
            name = ''
            date = ''
            for cell in value_cells:
                text = cell.get_text(strip=True)
                if text and not name:
                    name = text
                elif text and not date:
                    date = text
                # ignore extra cells
            if name:
                extracted.append({'head': 'Rapporteur name', 'value': name})
            if date:
                extracted.append({'head': 'Rapporteur date appointed', 'value': date})

        elif 'rapporteur for the opinion' in head_lower:
            # Opinion rapporteur
            name = ''
            date = ''
            for cell in value_cells:
                text = cell.get_text(strip=True)
                if text and not name:
                    name = text
                elif text and not date:
                    date = text
            if name:
                extracted.append({'head': 'Rapporteur opinion name', 'value': name})
            if date:
                extracted.append({'head': 'Rapporteur opinion date appointed', 'value': date})

        elif any(x in head_lower for x in ['members present', 'substitutes present']):
            # These rows contain lists of names. Collect all text from all value cells,
            # then split by commas or just take the whole text and clean.
            all_text = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            # Clean multiple spaces, commas, etc. Keep as is.
            # But we need to ensure names are separated by commas.
            # The text may already have commas; if not, we can split by common patterns.
            # For simplicity, we'll just store the whole cleaned text.
            extracted.append({'head': head, 'value': all_text})

        elif head_lower in ['title', 'committee responsible', 'opinion by', 'committee(s) asked for opinion(s)',
                            'discussed in committee', 'date adopted']:
            # Simple single value: collect text from all value cells and join.
            value = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            extracted.append({'head': head, 'value': value})

        else:
            # For any other row, we still capture it? Maybe skip. But the user might want all.
            # We'll capture anyway.
            value = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            if value:
                extracted.append({'head': head, 'value': value})

    return extracted


def expand_committee_procedure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the DataFrame from get_committee_votes_with_html and expands the procedure_string
    into a long-format table with columns: procedure, committee, designation, head, value.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: procedure, committee, designation, procedure_available, procedure_string.

    Returns
    -------
    pd.DataFrame
        Long-format table with extracted head-value pairs.
    """
    rows = []
    for _, row in df.iterrows():
        if not row['procedure_available']:
            continue
        html = row['procedure_string']
        if not html or pd.isna(html):
            continue
        # Determine committee type based on designation
        ctype = 'lead' if row['designation'] == 'lead' else 'opinion'
        extracted = parse_procedure_table(html, committee_type=ctype)
        for item in extracted:
            rows.append({
                'procedure': row['procedure'],
                'committee': row['committee'],
                'designation': row['designation'],
                'head': item['head'],
                'value': item['value']
            })
    return pd.DataFrame(rows)




In [10]:
df = pd.read_csv('bld\\test_procedures_committee_vote_info.csv')

In [34]:
from bs4 import BeautifulSoup
import pandas as pd
import re

def parse_procedure_table(html_str: str, committee_type: str = 'lead') -> list:
    """
    Parse the procedure table HTML and extract relevant rows as head-value pairs.
    Handles multi-line heads and multiple value cells.

    Parameters
    ----------
    html_str : str
        The HTML content of the div.table-responsive containing the procedure table.
    committee_type : str
        Either 'lead' or 'opinion'. Used for context, but not heavily used now.

    Returns
    -------
    list of dict
        Each dict has keys 'head' and 'value'.
    """
    soup = BeautifulSoup(html_str, 'html.parser')
    table = soup.find('table')
    if not table:
        return []

    rows = table.find_all('tr')
    extracted = []

    for tr in rows:
        cells = tr.find_all(['td', 'th'])
        if len(cells) < 2:
            continue

        # Head cell (first cell)
        head_cell = cells[0]
        head_text = head_cell.get_text('\n', strip=True)  # preserve line breaks
        # Split head by lines, ignore empty
        head_parts = [h.strip() for h in head_text.split('\n') if h.strip()]

        # Value cells (all after first)
        value_cells = cells[1:]

        # --- Special handling for result of final vote ---
        if any('result of final vote' in part.lower() for part in head_parts):
            symbols = []
            numbers = []
            # Collect lines from all value cells
            for cell in value_cells:
                cell_text = cell.get_text('\n', strip=True)
                lines = [ln.strip() for ln in cell_text.split('\n') if ln.strip()]
                for line in lines:
                    # Check if line is a symbol (maybe with colon)
                    if re.match(r'^[+\-–0]:?$', line):
                        sym = line.rstrip(':')
                        symbols.append(sym)
                    elif line.isdigit():
                        numbers.append(line)
                    # ignore any other content (should not happen)
            # Pair symbols and numbers in order
            for sym, num in zip(symbols, numbers):
                extracted.append({
                    'head': f'Result of final vote {sym}',
                    'value': num
                })
            continue

        # --- Special handling for members/substitutes (keep as one row) ---
        if any(key in head_text.lower() for key in ['members present', 'substitutes present']):
            all_text = ' '.join(cell.get_text(separator=' ', strip=True) for cell in value_cells)
            # Clean up multiple spaces, but keep names as a block
            all_text = re.sub(r'\s+', ' ', all_text).strip()
            extracted.append({'head': head_text, 'value': all_text})
            continue

        # --- General case: align head parts with value parts ---
        # Collect all text from value cells, each cell may contain multiple lines
        value_parts = []
        for cell in value_cells:
            cell_text = cell.get_text('\n', strip=True)
            parts = [p.strip() for p in cell_text.split('\n') if p.strip()]
            value_parts.extend(parts)

        # If number of head parts equals number of value parts, zip them
        if len(head_parts) == len(value_parts):
            for h, v in zip(head_parts, value_parts):
                extracted.append({'head': h, 'value': v})
        else:
            # Possible cases:
            # 1. One head part, multiple value parts (e.g., list of committees in separate cells)
            #    We'll treat each value part as a separate row with same head.
            if len(head_parts) == 1 and len(value_parts) > 1:
                for v in value_parts:
                    extracted.append({'head': head_parts[0], 'value': v})
            # 2. Multiple head parts, but value parts are grouped per cell (e.g., committees with dates)
            #    We need to pair head parts with groups of value parts from each cell.
            #    For example, head = ["Committee asked", "Date announced"] and each value cell contains two lines.
            #    We'll assume each value cell has as many lines as head_parts.
            elif len(head_parts) > 1:
                # Process each value cell separately
                for cell in value_cells:
                    cell_lines = cell.get_text('\n', strip=True).split('\n')
                    cell_lines = [ln.strip() for ln in cell_lines if ln.strip()]
                    if len(cell_lines) == len(head_parts):
                        for h, v in zip(head_parts, cell_lines):
                            extracted.append({'head': h, 'value': v})
                    else:
                        # fallback: treat whole cell as one value for each head? Not good.
                        # Instead, just join and give one row per head? We'll skip for now.
                        pass
            else:
                # Fallback: just join all value parts and assign to first head
                extracted.append({'head': head_parts[0] if head_parts else head_text,
                                  'value': ' '.join(value_parts)})

    return extracted


def expand_committee_procedure(df: pd.DataFrame) -> pd.DataFrame:
    """
    Takes the DataFrame from get_committee_votes_with_html and expands the procedure_string
    into a long-format table with columns: procedure, committee, designation, head, value.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain columns: procedure, committee, designation, procedure_available, procedure_string.

    Returns
    -------
    pd.DataFrame
        Long-format table with extracted head-value pairs.
    """
    rows = []
    for _, row in df.iterrows():
        if not row['procedure_available']:
            continue
        html = row['procedure_string']
        if not html or pd.isna(html):
            continue
        # Determine committee type based on designation
        ctype = 'lead' if row['designation'] == 'lead' else 'opinion'
        extracted = parse_procedure_table(html, committee_type=ctype)
        for item in extracted:
            rows.append({
                'procedure': row['procedure'],
                'committee': row['committee'],
                'designation': row['designation'],
                'head': item['head'],
                'value': item['value']
            })
    return pd.DataFrame(rows)




In [45]:
df_long = expand_committee_procedure(df)

df_long.head(50)

,procedure,committee,designation,head,value
0,2021/0366(COD),ENVI,lead,Title,Making available on the Union market as well a...
1,2021/0366(COD),ENVI,lead,References,COM(2021)0706
2,2021/0366(COD),ENVI,lead,References,– C9-0430/2021 –
3,2021/0366(COD),ENVI,lead,References,2021/0366(COD)
4,2021/0366(COD),ENVI,lead,Date submitted to Parliament,18.11.2021
5,2021/0366(COD),ENVI,lead,Committee responsible,ENVI
6,2021/0366(COD),ENVI,lead,Date announced in plenary,17.1.2022
7,2021/0366(COD),ENVI,lead,Committees asked for opinions,DEVE
8,2021/0366(COD),ENVI,lead,Date announced in plenary,7.4.2022
9,2021/0366(COD),ENVI,lead,Committees asked for opinions,INTA
